In [9]:
import os
import duckdb
import pandas as pd

# 1. Definir rutas
dataset_path = ""
dataset_filename = "competencia_02_crudo.csv.gz"
output_filename = "competencia_02.csv"

file_path = os.path.join(dataset_path, dataset_filename)
output_path = os.path.join(dataset_path, output_filename)

# 2. Inicializar conexión a DuckDB
con = duckdb.connect(database=":memory:")

print("Cargando dataset crudo y calculando la clase ternaria (con tolerancia a errores de formato)...")

# 3. Ejecutar la query con null_padding e ignore_errors
con.execute(f"""
    CREATE OR REPLACE TABLE competencia_02 AS
    WITH periodos AS (
        SELECT DISTINCT foto_mes FROM read_csv_auto('{file_path}', null_padding=true, ignore_errors=true)
    ),
    clientes AS (
        SELECT DISTINCT numero_de_cliente FROM read_csv_auto('{file_path}', null_padding=true, ignore_errors=true)
    ),
    todo AS (
        SELECT numero_de_cliente, foto_mes FROM clientes CROSS JOIN periodos
    ),
    clase_ternaria AS (
        SELECT
            c.*
            , IF(c.numero_de_cliente IS NULL, 0, 1) AS mes_0
            , LEAD(mes_0, 1) OVER (PARTITION BY t.numero_de_cliente ORDER BY foto_mes) AS mes_1
            , LEAD(mes_0, 2) OVER (PARTITION BY t.numero_de_cliente ORDER BY foto_mes) AS mes_2
            , CASE
                WHEN mes_0 = 1 AND mes_1 = 0 THEN 'BAJA+1'
                WHEN mes_0 = 1 AND mes_1 = 1 AND mes_2 = 0 THEN 'BAJA+2'
                WHEN mes_0 = 1 AND mes_1 = 1 AND mes_2 = 1 THEN 'CONTINUA'
                ELSE NULL
              END AS clase_ternaria
        FROM todo t
        LEFT JOIN read_csv_auto('{file_path}', null_padding=true, ignore_errors=true) c USING (numero_de_cliente, foto_mes)
    )
    SELECT * EXCLUDE (mes_0, mes_1, mes_2)
    FROM clase_ternaria
    WHERE mes_0 = 1
""")

print(f"Guardando el resultado procesado en '{output_path}'...")

# 4. Exportar la tabla resultante a un nuevo archivo CSV
con.execute(f"""
    COPY competencia_02 TO '{output_path}' (FORMAT CSV, HEADER)
""")

print("\n¡Proceso finalizado con éxito y clase ternaria creada!")

Cargando dataset crudo y calculando la clase ternaria (con tolerancia a errores de formato)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Guardando el resultado procesado en 'competencia_02.csv'...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


¡Proceso finalizado con éxito y clase ternaria creada!


In [11]:
import pandas as pd

print("1. Cargando el dataset con la clase ternaria...")
df = pd.read_csv('competencia_02.csv')

print("2. Ordenando estrictamente por cliente y mes...")
df = df.sort_values(by=['numero_de_cliente', 'foto_mes']).reset_index(drop=True)

cols_excluir = ['numero_de_cliente', 'foto_mes', 'clase_ternaria']
features = [c for c in df.columns if c not in cols_excluir]

print(f"3. Calculando lags y deltas de forma eficiente para {len(features)} variables...")
nuevas_columnas = {}
grouped = df.groupby('numero_de_cliente')

for col in features:
    if pd.api.types.is_numeric_dtype(df[col]):
        # Obtenemos la serie agrupada para este col
        s_col = grouped[col]
        lag1 = s_col.shift(1).fillna(df[col])
        lag2 = s_col.shift(2).fillna(df[col])
        
        # Guardamos en el diccionario en lugar de modificar el df original
        nuevas_columnas[f'lag1_{col}'] = lag1
        nuevas_columnas[f'lag2_{col}'] = lag2
        nuevas_columnas[f'delta1_{col}'] = df[col] - lag1
        nuevas_columnas[f'delta2_{col}'] = df[col] - lag2

print("4. Uniendo todas las columnas nuevas al DataFrame...")
df_nuevas = pd.DataFrame(nuevas_columnas)
df = pd.concat([df, df_nuevas], axis=1)

print("5. Guardando el resultado final...")
df.to_csv('competencia_02_fe.csv', index=False)
print("¡Listo! Feature engineering finalizado sin fragmentación y guardado con éxito.")

1. Cargando el dataset con la clase ternaria...
2. Ordenando estrictamente por cliente y mes...
3. Calculando lags y deltas de forma eficiente para 152 variables...
4. Uniendo todas las columnas nuevas al DataFrame...
5. Guardando el resultado final...
¡Listo! Feature engineering finalizado sin fragmentación y guardado con éxito.
